# 18. 기준일 이후 추가 상승분(post-boost) — 9월 첫 주 실제 조회수 흐름
기준일 D 직전 5.5일 영상의 view_5d는 D **이후**에 측정됨. 같은 영상의 D 직전 스냅샷 조회수와 비교하면
"D 이후 이 채널 영상이 평소보다 얼마나 더 보였나"를 채널별로 직접 측정할 수 있다 (09-01에선 9월 첫 주).
- 기대값 = D 직전 마지막 스냅샷 × (그 나이 → 5일 시점 평소 증가 비율; D 이전 30일 영상으로 학습)
- 상승분 = log view_5d − log 기대값, 날짜별 중앙값을 빼 공통 효과(급등) 제거
학습 기준일에도 같은 구조 → 학습·검증 가능.

In [1]:
import pickle, duckdb, numpy as np, pandas as pd
from scipy.stats import rankdata
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score
dd = pickle.load(open('../data/processed/_refine_cache.pkl', 'rb'))
po, test, FC, INTER = dd['po'], dd['test'], dd['FC'], dd['INTER']
con = duckdb.connect()
con.sql("SET memory_limit='6GB'")
con.sql("CREATE VIEW v5 AS SELECT * FROM read_csv('../data/raw/epoch_data/video_5d_views.csv', header=true, auto_detect=true)")
con.sql("CREATE VIEW sn AS SELECT * FROM '../data/processed/video_snapshots.parquet'")

def post_boost(D):
    return con.sql(f"""
    WITH hv AS (SELECT video_id, LN(1 + view_5d) l5 FROM v5 WHERE is_shorts = 0 AND view_5d IS NOT NULL
                AND published_at >= TIMESTAMP '{D}' - INTERVAL 30 DAY AND published_at < TIMESTAMP '{D}' - INTERVAL 132 HOUR),
    hs AS (SELECT s.video_id, CAST(FLOOR(s.video_age_hours / 6) AS INT) b, ARG_MAX(s.view_count, s.collected_at) vc
           FROM sn s JOIN hv USING (video_id) WHERE s.video_age_hours < 132 GROUP BY 1, 2),
    ratio AS (SELECT b, MEDIAN(hv.l5 - LN(1 + hs.vc)) lr FROM hs JOIN hv USING (video_id) GROUP BY b),
    tv AS (SELECT video_id, channel_id, LN(1 + view_5d) l5 FROM v5 WHERE is_shorts = 0 AND view_5d IS NOT NULL
           AND published_at >= TIMESTAMP '{D}' - INTERVAL 132 HOUR AND published_at < TIMESTAMP '{D}'),
    last AS (SELECT s.video_id, ARG_MAX(s.view_count, s.collected_at) vc, MAX(s.video_age_hours) age
             FROM sn s JOIN tv USING (video_id) WHERE s.collected_at < TIMESTAMP '{D}' GROUP BY 1)
    SELECT tv.channel_id, AVG(tv.l5 - (LN(1 + last.vc) + ratio.lr)) AS boost, COUNT(*) AS n_boost,
           MAX(tv.l5 - (LN(1 + last.vc) + ratio.lr)) AS boost_max
    FROM tv JOIN last USING (video_id) JOIN ratio ON ratio.b = CAST(FLOOR(last.age / 6) AS INT)
    GROUP BY 1""").df()

D = '2026-09-01'
dates = sorted(po.ref_date.unique())
pb = []
for d in dates + [D]:
    x = post_boost(d).assign(ref_date=d)
    med = x.boost.median()
    x['boost_c'] = x.boost - med; x['boost_max_c'] = x.boost_max - med
    pb.append(x); print(d, '채널', len(x), '| 공통 상승분(중앙값)', round(med, 3))
pb = pd.concat(pb, ignore_index=True)

2026-07-26 채널 224 | 공통 상승분(중앙값) 0.024


2026-07-27 채널 217 | 공통 상승분(중앙값) 0.02


2026-07-28 채널 613 | 공통 상승분(중앙값) 0.01


2026-07-29 채널 602 | 공통 상승분(중앙값) 0.005


2026-07-30 채널 596 | 공통 상승분(중앙값) 0.002


...

2026-08-02 채널 599 | 공통 상승분(중앙값) 0.016


2026-08-03 채널 615 | 공통 상승분(중앙값) 0.022


2026-08-04 채널 603 | 공통 상승분(중앙값) 0.011


2026-08-05 채널 599 | 공통 상승분(중앙값) 0.005


2026-09-01 채널 622 | 공통 상승분(중앙값) 0.009


In [2]:
BF = ['boost_c', 'boost_max_c', 'n_boost']
po2 = po.merge(pb[['channel_id', 'ref_date'] + BF], on=['channel_id', 'ref_date'], how='left'); po2.index = po.index
te2 = test.merge(pb[pb.ref_date == D][['channel_id'] + BF], on='channel_id', how='left'); te2.index = test.index
po2['n_boost'] = po2.n_boost.fillna(0); te2['n_boost'] = te2.n_boost.fillna(0)
print('boost 보유율: 학습', round(po2.boost_c.notna().mean(), 3), '| 테스트', round(te2.boost_c.notna().mean(), 3))
print('단독 PR-AUC (전주있음 행, 날짜평균):',
      round(np.mean([average_precision_score(g.target, g.boost_c.fillna(g.boost_c.median())) for _, g in po2[po2.prev.notna()].groupby('ref_date')]), 4))

logreg = lambda C: Pipeline([('imp', SimpleImputer(strategy='median')), ('sc', StandardScaler()),
                             ('m', LogisticRegression(max_iter=3000, C=C, class_weight='balanced'))])
def cv(cols, seeds=(0, 1)):
    out = []
    for seed in seeds:
        ch = po2.channel_id.values; u = np.unique(ch); rg = np.random.default_rng(seed); rg.shuffle(u)
        fold = np.array([dict(zip(u, np.arange(len(u)) % 5))[c] for c in ch]); p = np.zeros(len(po2))
        for f in range(5):
            p[fold == f] = logreg(0.01).fit(po2[fold != f][cols], po2[fold != f].target).predict_proba(po2[fold == f][cols])[:, 1]
        s = pd.Series(p, index=po2.index)
        out.append(np.mean([average_precision_score(g.target, s.loc[g.index]) for _, g in po2[po2.prev.notna()].groupby('ref_date')]))
    return round(np.mean(out), 4)
print('V1 (FC+INTER)          :', cv(FC + INTER))
print('V1 + post-boost         :', cv(FC + INTER + BF))

boost 보유율: 학습 0.849 | 테스트 0.794
단독 PR-AUC (전주있음 행, 날짜평균): 0.2387


V1 (FC+INTER)          : 0.5809


V1 + post-boost         : 0.5817


### 결과: 09-01 공통 상승분 0.009 (≈0), 피처 추가 효과 없음(0.5809→0.5817). 급등은 08-31 이전 스냅샷에 이미 보이는 실제 상승.

## 과거 기준선 부풀림(d_c)과 8/25 라벨의 관계
정답 = log(9월 중앙값) − log(9/1 이전 중앙값, 원값). 9월은 모든 채널 공통 상승 → 상쇄. 8월 말 업로드가 많은 채널은 **과거 중앙값이 부풀어(d_c)** growth가 깎임.
그런데 8/25 라벨은 부풀린 08-25~31 영상을 정답 구간에 포함 → 바로 이 채널들을 라벨=1로 만들었을 가능성. 38에서 이들의 위치 확인.

In [3]:
sample = pd.read_csv('../data/raw/epoch_data/sample_submission.csv', encoding='utf-8-sig')
sample['channel_id'] = sample.row_id.str.rsplit('_', n=1).str[0]
con.sql("CREATE OR REPLACE TABLE dl AS SELECT * FROM (VALUES " + ",".join(
    f"(DATE '2026-08-{d:02d}', {s})" for d, s in [(24, .045), (25, .092), (26, .326), (27, .661), (28, .759), (29, .804), (30, .789), (31, .782)]) + ") t(day, s)")
pm = con.sql("""SELECT channel_id, MEDIAN(LN(1 + view_5d)) lp_raw, MEDIAN(LN(1 + view_5d) - COALESCE(dl.s, 0)) lp_def,
                       COUNT(*) FILTER (WHERE published_at >= TIMESTAMP '2026-08-26') n_late, COUNT(*) n_all
                FROM v5 LEFT JOIN dl ON CAST(published_at AS DATE) = dl.day
                WHERE is_shorts = 0 AND view_5d IS NOT NULL AND published_at < TIMESTAMP '2026-09-01' GROUP BY 1""").df()
X = sample.merge(pm, on='channel_id', how='left')
X['d_c'] = X.lp_raw - X.lp_def
tl = pd.read_csv('../data/raw/epoch_data/train_labels.csv'); tl['channel_id'] = tl.row_id.str.rsplit('_', n=1).str[0]
X = X.merge(tl[tl.row_id.str.endswith('2026-08-25')][['channel_id', 'target']].rename(columns={'target': 'prev'}), on='channel_id', how='left')
X['r38'] = rankdata(pd.read_csv('../submissions/submit_38_ens21_prevrule50.csv').set_index('row_id').prediction.loc[X.row_id].values) / len(X)
X['grp'] = pd.cut(X.d_c, [-1, 0.02, 0.1, 0.3, 5], labels=['0 (부풀림 없음)', '0.02~0.1', '0.1~0.3', '0.3+'])
print('d_c 구간별: 채널 수 / 8/25 라벨=1 비율 / 38 평균 순위 / 38 상위139 포함 수')
top = set(X.nlargest(139, 'r38').index)
print(X.groupby('grp', observed=True).apply(lambda g: pd.Series({'n': len(g), '라벨=1 비율': round((g.prev == 1).mean(), 3),
      '38 평균순위': round(g.r38.mean(), 3), '38 상위139': len(set(g.index) & top)})).to_string())

d_c 구간별: 채널 수 / 8/25 라벨=1 비율 / 38 평균 순위 / 38 상위139 포함 수
                n  라벨=1 비율  38 평균순위  38 상위139
grp                                          
0 (부풀림 없음)  307.0    0.254    0.567      84.0
0.02~0.1    196.0    0.148    0.420      28.0
0.1~0.3     144.0    0.132    0.464      21.0
0.3+         47.0    0.170    0.520       6.0


### 결과: 8/25 라벨=1 비율은 부풀림 없는 채널에서 가장 높음(0.254), 38도 이미 이들을 위에 둠 → 바로잡을 왜곡 없음.

## 새 축: 채널 간 정보 공유 (같은 게임·주제 채널들의 전주 라벨 평균)
같은 게임을 다루는 채널은 함께 움직인다. 이웃 채널들의 8/25 라벨 = 그 게임/주제가 9월 초에 떴는지에 대한 신호.
학습쌍(D−7 라벨)에도 같은 구조 → 검증 가능. 자기 자신은 제외한 평균.

In [4]:
RAW = '../data/raw/epoch_data'
for t in ['videos', 'video_igdb_games', 'video_games', 'channels']:
    con.sql(f"CREATE OR REPLACE VIEW {t} AS SELECT * FROM read_csv('{RAW}/{t}.csv', header=true, auto_detect=true)")

def chan_games(D):
    return con.sql(f"""
      SELECT DISTINCT v.channel_id, g.gid FROM videos v JOIN (
          SELECT CAST(video_id AS VARCHAR) vid, 'i' || CAST(igdb_id AS VARCHAR) gid FROM video_igdb_games
          UNION ALL SELECT CAST(video_id AS VARCHAR), 's' || CAST(app_id AS VARCHAR) FROM video_games) g
        ON CAST(v.video_id AS VARCHAR) = g.vid
      WHERE v.published_at < TIMESTAMP '{D}' AND v.published_at >= TIMESTAMP '{D}' - INTERVAL 45 DAY""").df()
ch_meta = con.sql("SELECT channel_id, discovery_keyword kw, topic_category tc FROM channels").df()

def neighbor_feats(df, D):
    """df: channel_id, prev (해당 날짜 행들). 같은 게임/키워드/주제 채널들의 prev 평균 (자기 제외)"""
    base = df[['channel_id', 'prev']].copy()
    out = pd.DataFrame({'channel_id': base.channel_id})
    cg = chan_games(D).merge(base, on='channel_id', how='inner')
    s = cg.dropna(subset=['prev']).groupby('gid').prev.agg(['sum', 'count'])
    cg = cg.join(s, on='gid')
    own = cg.prev.fillna(0); own_c = cg.prev.notna().astype(int)
    cg['ns'] = cg['sum'].fillna(0) - own; cg['nc'] = cg['count'].fillna(0) - own_c
    agg = cg.groupby('channel_id')[['ns', 'nc']].sum()
    out = out.merge(agg, on='channel_id', how='left')
    out['nb_game_prev'] = out.ns / out.nc.where(out.nc > 0)
    out['nb_game_n'] = np.log1p(out.nc.fillna(0))
    for col in ['kw', 'tc']:
        m = base.merge(ch_meta[['channel_id', col]], on='channel_id', how='left')
        grp = m.dropna(subset=['prev']).groupby(col).prev.agg(['sum', 'count'])
        m = m.join(grp, on=col)
        own = m.prev.fillna(0); own_c = m.prev.notna().astype(int)
        m[f'nb_{col}_prev'] = (m['sum'] - own) / (m['count'] - own_c).where((m['count'] - own_c) > 0)
        out = out.merge(m[['channel_id', f'nb_{col}_prev']], on='channel_id', how='left')
    return out[['channel_id', 'nb_game_prev', 'nb_game_n', 'nb_kw_prev', 'nb_tc_prev']]

NB = ['nb_game_prev', 'nb_game_n', 'nb_kw_prev', 'nb_tc_prev']
parts = []
for d in dates:
    sub_ = po2[po2.ref_date == d]
    parts.append(sub_[['channel_id']].merge(neighbor_feats(sub_, d), on='channel_id', how='left').set_index(sub_.index))
po3 = po2.join(pd.concat(parts)[NB])
te3 = te2.join(te2[['channel_id']].merge(neighbor_feats(te2, D), on='channel_id', how='left').set_index(te2.index)[NB])
print('보유율 학습:', po3[NB].notna().mean().round(3).to_dict())
print('보유율 테스트:', te3[NB].notna().mean().round(3).to_dict())
po2 = po3
for c in NB[:1] + NB[2:]:
    print(c, '단독 PR-AUC (전주있음 행):', round(np.mean([average_precision_score(g.target, g[c].fillna(g[c].median()).fillna(0)) for _, g in po2[po2.prev.notna()].groupby('ref_date')]), 4))
print('V1 (FC+INTER)        :', cv(FC + INTER))
print('V1 + 이웃 신호 4     :', cv(FC + INTER + NB))
print('V1 + 게임 이웃만      :', cv(FC + INTER + ['nb_game_prev', 'nb_game_n']))

보유율 학습: {'nb_game_prev': 0.59, 'nb_game_n': 1.0, 'nb_kw_prev': 0.0, 'nb_tc_prev': 0.986}
보유율 테스트: {'nb_game_prev': 0.674, 'nb_game_n': 1.0, 'nb_kw_prev': 0.0, 'nb_tc_prev': 0.993}
nb_game_prev 단독 PR-AUC (전주있음 행): 0.2265
nb_kw_prev 단독 PR-AUC (전주있음 행): 0.1936
nb_tc_prev 단독 PR-AUC (전주있음 행): 0.167


V1 (FC+INTER)        : 0.5809


C:\Users\gjymo\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\sklearn\impute\_base.py:647: UserWarning: Skipping features without any observed values: ['nb_kw_prev']. At least one non-missing value is needed for imputation with strategy='median'.
  warnings.warn(
C:\Users\gjymo\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\sklearn\impute\_base.py:647: UserWarning: Skipping features without any observed values: ['nb_kw_prev']. At least one non-missing value is needed for imputation with strategy='median'.
  warnings.warn(
C:\Users\gjymo\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\sklearn\impute\_base.py:647: UserWarning: Skipping features without any observed values: ['nb_kw_prev']. At least one non-missing value is needed for imputation with strategy='median'.
  warnings.warn(
C:\Users\gjymo\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\sklearn\impute\_base.py:647: UserWarning: Skipping features without any observed values: ['nb_kw_prev'

C:\Users\gjymo\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\sklearn\impute\_base.py:647: UserWarning: Skipping features without any observed values: ['nb_kw_prev']. At least one non-missing value is needed for imputation with strategy='median'.
  warnings.warn(
C:\Users\gjymo\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\sklearn\impute\_base.py:647: UserWarning: Skipping features without any observed values: ['nb_kw_prev']. At least one non-missing value is needed for imputation with strategy='median'.
  warnings.warn(
C:\Users\gjymo\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\sklearn\impute\_base.py:647: UserWarning: Skipping features without any observed values: ['nb_kw_prev']. At least one non-missing value is needed for imputation with strategy='median'.
  warnings.warn(
C:\Users\gjymo\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\sklearn\impute\_base.py:647: UserWarning: Skipping features without any observed values: ['nb_kw_prev'

V1 + 이웃 신호 4     : 0.5807


V1 + 게임 이웃만      : 0.5804
